# Exchange-Programme Bulletin Router

The `data/` folder holds four editions of the same recurring Tunghai bulletin
(*全球姊妹校交換計畫甄選簡章*), spring and autumn, for 2024 and 2025.

- **2024** editions are loaded into the knowledge base.
- **2025** editions play the part of a document a user uploads.

These four PDFs are **93-95% identical to one another**, so "is this an update?" is not a
useful question - the answer is always yes. The useful question is **which sections
actually changed, and did the substance change or only the year label**. So the routing
happens per *section*, not per document.


In [1]:
import re
import sys
from pathlib import Path

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA_DIR = PROJECT_ROOT / "data"
sys.path.insert(0, str(PROJECT_ROOT))   # so we can reuse extract.py

KB_YEAR = "2024"        # editions that seed the knowledge base
INCOMING_YEAR = "2025"  # editions treated as user uploads

SEPARATORS = ["\n\n", "\n", "。", "！", "？", "；",
              ". ", "! ", "? ", "; ", "!", "?", ";", " ", ""]

EMBED_MODEL = "intfloat/multilingual-e5-base"
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"

CHUNK_SIZE_CHARS = 800
CHUNK_OVERLAP_CHARS = 200

COLLECTION = "oir_bulletins"

TOP_K = 5          # KB candidates recalled per incoming section
BATCH_SIZE = 32

# Section-level decision thresholds.
RELEVANCE_MIN = 0.30   # reranker prob below this => no counterpart in the KB => NEW
IDENTICAL_MIN = 0.95   # share of the section already present in the KB => UNCHANGED

print("Project root:", PROJECT_ROOT)


Project root: C:\Users\David Gunawan Wisno\Documents\project\final-project\oir-hub-ai


In [2]:
import torch
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_text_splitters import RecursiveCharacterTextSplitter

import os
from extract import extract_text   # reuses the same extractor the FastAPI app uses

load_dotenv(PROJECT_ROOT / ".env")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")


C:\Users\David Gunawan Wisno\Documents\project\final-project\oir-hub-ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda NVIDIA GeForce RTX 3070 Laptop GPU


### Read the PDFs

`extract.py` already knows how to pull text out of a PDF (and .docx, and plain text), so the
notebook and the API cannot drift apart.

One addition on top of it: a 12-page PDF repeats its running header and footer on every
page. Left in, that furniture becomes a dozen near-identical chunks that match each other
far better than they match anything meaningful, which pollutes retrieval. So lines that
recur across most pages get dropped.


In [3]:
from collections import Counter

def strip_page_furniture(text, min_repeats=4, max_len=120):
    """Drop running headers/footers - lines that recur on most pages carry no signal.

    max_len must clear the longest banner: these bulletins carry a 71-character bilingual
    header on all 12 pages, and a tighter cap silently leaves it in every chunk."""
    lines = text.splitlines()
    # normalise the page number so "... P1" and "... P2" count as the same line
    norm = lambda l: re.sub(r"P\s*\d+\s*$", "", l.strip())
    counts = Counter(norm(l) for l in lines if l.strip())
    kept = [l for l in lines
            if not (norm(l) and counts[norm(l)] >= min_repeats and len(norm(l)) <= max_len)]
    return "\n".join(kept)


def load_pdf(path: Path) -> str:
    raw = extract_text(path.name, path.read_bytes())
    return strip_page_furniture(raw)


def edition_of(path: Path):
    """Pull ('2024', 'spring') out of a filename like 2024年春....pdf"""
    year = re.search(r"(20\d{2})\s*年", path.name)
    term = "spring" if "春" in path.name else "autumn" if "秋" in path.name else "unknown"
    return (year.group(1) if year else "?"), term


documents = {}
for path in sorted(DATA_DIR.glob("*.pdf")):
    year, term = edition_of(path)
    text = load_pdf(path)
    documents[path.name] = {"path": path, "year": year, "term": term, "text": text}
    print(f"{year} {term:<7} {len(text):>6} chars   {path.name}")

kb_files       = [n for n, d in documents.items() if d["year"] == KB_YEAR]
incoming_files = [n for n, d in documents.items() if d["year"] == INCOMING_YEAR]
print(f"\nknowledge base : {len(kb_files)} file(s)")
print(f"incoming       : {len(incoming_files)} file(s)")


2024 spring    9555 chars   2024年春全球姊妹校交換計畫甄選簡章_1012.pdf


2024 autumn    8983 chars   2024年秋全球姊妹校交換計畫甄選簡章_0314.pdf


2025 spring    9514 chars   2025年春全球姊妹校交換計畫甄選簡章_10-01.pdf


2025 autumn   10029 chars   2025年秋全球姊妹校交換計畫甄選簡章_0319.pdf

knowledge base : 2 file(s)
incoming       : 2 file(s)


#### How alike are these editions?

Worth looking at before trusting any similarity score. This is the whole difficulty of the
task in one table.


In [4]:
import difflib

names = list(documents)
print(f"{'':<14}" + "".join(f"{documents[n]['year']}{documents[n]['term'][:3]:>4}  " for n in names))
for a in names:
    row = "".join(
        f"{difflib.SequenceMatcher(None, documents[a]['text'], documents[b]['text']).quick_ratio():>9.3f} "
        for b in names
    )
    print(f"{documents[a]['year']}{documents[a]['term'][:3]:<9}" + row)

print("\nNote: 2025 spring is closer to 2024 autumn than to 2024 spring, so 'most similar")
print("document wins' would pair them up wrongly. Section-level matching avoids the trap.")


              2024 spr  2024 aut  2025 spr  2025 aut  
2024spr          1.000     0.938     0.936     0.917 
2024aut          0.938     1.000     0.945     0.926 
2025spr          0.936     0.945     1.000     0.937 
2025aut          0.917     0.926     0.937     1.000 

Note: 2025 spring is closer to 2024 autumn than to 2024 spring, so 'most similar
document wins' would pair them up wrongly. Section-level matching avoids the trap.


In [5]:
client = QdrantClient(url=os.getenv("QDRANT_URL"), api_key=os.getenv("QDRANT_API_KEY"))

model_kwargs = {"torch_dtype": torch.float16} if DEVICE == "cuda" else {}
model = SentenceTransformer(
    EMBED_MODEL, device=DEVICE, token=os.getenv("HUGGINGFACE_TOKEN"), model_kwargs=model_kwargs
)
VECTOR_SIZE = model.get_embedding_dimension()

reranker = CrossEncoder(RERANK_MODEL, device=DEVICE, max_length=512)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE_CHARS,
    chunk_overlap=CHUNK_OVERLAP_CHARS,
    separators=SEPARATORS,
    keep_separator="end",
)

print("Embedder:", EMBED_MODEL, f"({VECTOR_SIZE}d)")
print("Reranker:", RERANK_MODEL)


C:\Users\David Gunawan Wisno\AppData\Local\Temp\ipykernel_27600\3671878914.py:1: UserWarning: Api key is used with an insecure connection.
  client = QdrantClient(url=os.getenv("QDRANT_URL"), api_key=os.getenv("QDRANT_API_KEY"))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   3%|▎         | 5/199 [00:00<00:04, 45.33it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1400.62it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 8260.43it/s]

Embedder: intfloat/multilingual-e5-base (768d)
Reranker: BAAI/bge-reranker-v2-m3


#### Add Prefix

e5 is asymmetric: stored text needs `passage: `, search text needs `query: `.


In [6]:
def embed_passages(texts):
    return model.encode([f"passage: {t}" for t in texts],
                        normalize_embeddings=True, batch_size=BATCH_SIZE)

def embed_queries(texts):
    return model.encode([f"query: {t}" for t in texts],
                        normalize_embeddings=True, batch_size=BATCH_SIZE)


### Load the 2024 editions into Qdrant


In [7]:
import uuid
from qdrant_client.models import (
    Distance, VectorParams, PointStruct, PayloadSchemaType, QueryRequest,
)

client.delete_collection(COLLECTION)
client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
)
for field in ("source_file", "year", "term"):
    client.create_payload_index(
        collection_name=COLLECTION, field_name=field, field_schema=PayloadSchemaType.KEYWORD
    )

# Chunk everything first, then embed and upsert once.
all_chunks, all_meta = [], []
for name in kb_files:
    doc = documents[name]
    chunks = splitter.split_text(doc["text"])
    all_chunks.extend(chunks)
    all_meta.extend((name, doc["year"], doc["term"], i) for i in range(len(chunks)))
    print(f"{name}: {len(chunks)} chunks")

vectors = embed_passages(all_chunks)
client.upsert(
    collection_name=COLLECTION,
    points=[
        PointStruct(
            id=str(uuid.uuid5(uuid.NAMESPACE_URL, f"{name}:{idx}")),
            vector=vec.tolist(),
            payload={"source_file": name, "year": year, "term": term,
                     "chunk_index": idx, "content": text},
        )
        for text, vec, (name, year, term, idx) in zip(all_chunks, vectors, all_meta)
    ],
    wait=True,
)
print("\nTotal points:", client.count(COLLECTION).count)


2024年春全球姊妹校交換計畫甄選簡章_1012.pdf: 18 chunks
2024年秋全球姊妹校交換計畫甄選簡章_0314.pdf: 15 chunks



Total points: 33


### Compare an uploaded edition, section by section

For every section of the uploaded document: recall `TOP_K` candidates from the knowledge
base in one batched call, rerank them with the cross-encoder, then classify against the
best one.

- **UNCHANGED** - a counterpart exists and the wording is materially the same.
- **REVISED** - a counterpart exists but the wording moved. *These are what to review.*
- **NEW** - nothing in the knowledge base covers this.


In [8]:
def normalise(s):
    """Ignore whitespace and the year label so a pure 2024->2025 relabel is not 'revised'."""
    s = re.sub(r"20\d{2}", "<Y>", s)
    return re.sub(r"\s+", "", s)


def coverage(section, corpus):
    """How much of this section already exists in the KB, 0-1.

    Measured against the whole corpus, not one chunk: chunk boundaries shift between
    editions, so a chunk-to-chunk diff calls every section changed even when the wording
    is identical. autojunk must stay off - its 1%-frequency heuristic discards the ratio
    entirely on CJK text of this length.
    """
    a, b = normalise(section), normalise(corpus)
    if not a:
        return 1.0
    m = difflib.SequenceMatcher(None, a, b, autojunk=False)
    return sum(blk.size for blk in m.get_matching_blocks()) / len(a)


def analyse(name):
    doc = documents[name]
    sections = splitter.split_text(doc["text"])

    # one Qdrant round-trip for every section
    results = client.query_batch_points(
        collection_name=COLLECTION,
        requests=[QueryRequest(query=v.tolist(), limit=TOP_K, with_payload=True)
                  for v in embed_queries(sections)],
    )

    pairs, flat = [], []
    for i, res in enumerate(results):
        for hit in res.points:
            pairs.append((sections[i], hit.payload["content"]))
            flat.append((i, hit))

    scores = reranker.predict(pairs, activation_fn=torch.nn.Sigmoid(), batch_size=BATCH_SIZE)

    # keep the single best KB counterpart per section
    best = {}
    for (i, hit), score in zip(flat, scores):
        if i not in best or score > best[i]["rerank"]:
            best[i] = {"hit": hit, "rerank": float(score)}

    # every distinct KB chunk that surfaced, as one corpus to diff against
    seen, corpus = set(), []
    for _, hit in flat:
        if hit.id not in seen:
            seen.add(hit.id)
            corpus.append(hit.payload["content"])
    corpus = " ".join(corpus)

    # Coverage is checked before the reranker score, not after. The reranker judges
    # relevance, not near-duplication, and scores some boilerplate low even when it is in
    # the KB word for word - gating on it first files text that plainly exists as NEW.
    report = []
    for i, section in enumerate(sections):
        b = best.get(i)
        cov = coverage(section, corpus)
        if cov >= IDENTICAL_MIN:
            status = "UNCHANGED"
        elif b is not None and b["rerank"] >= RELEVANCE_MIN:
            status = "REVISED"
        else:
            status = "NEW"
        report.append({
            "index": i, "status": status, "section": section, "overlap": cov,
            "rerank": b["rerank"] if b else 0.0,
            "matched": b["hit"].payload["content"] if b else None,
            "match": (f"{b['hit'].payload['source_file']}#{b['hit'].payload['chunk_index']}"
                      if b and status != "NEW" else None),
        })
    return report


def summarise(name, report):
    counts = Counter(r["status"] for r in report)
    print(f"=== {name} ===")
    print(f"    {len(report)} sections -> "
          f"{counts['UNCHANGED']} unchanged, {counts['REVISED']} revised, {counts['NEW']} new")
    verdict = "UPDATE" if counts["REVISED"] or counts["NEW"] else "NO CHANGE"
    print(f"    verdict: {verdict}\n")
    for r in report:
        if r["status"] == "UNCHANGED":
            continue
        head = r["section"].strip().replace("\n", " ")[:70]
        extra = f"overlap {r['overlap']:.2f} vs {r['match']}" if r["match"] else "no counterpart"
        print(f"  [{r['status']:<9}] rerank {r['rerank']:.3f}  {extra}")
        print(f"              {head}...")
    return counts


reports = {name: analyse(name) for name in incoming_files}
for name in incoming_files:
    summarise(name, reports[name])


=== 2025年春全球姊妹校交換計畫甄選簡章_10-01.pdf ===
    18 sections -> 5 unchanged, 9 revised, 4 new
    verdict: UPDATE

  [REVISED  ] rerank 1.000  overlap 0.94 vs 2024年春全球姊妹校交換計畫甄選簡章_1012.pdf#3
              2025年春學期東海大學赴全球姊妹校交換甄試簡章    赴姊妹校交換學期：2025年春季學期(部分姊妹校2025年秋季)    甄試宗旨  ...
  [REVISED  ] rerank 0.986  overlap 0.85 vs 2024年春全球姊妹校交換計畫甄選簡章_1012.pdf#2
              重要日程(節錄)  2024年9月-10月  日期 事項 備註  9月2(一) 公告甄選簡章 14:00 開放網路報名  9月26日(四) ...
  [REVISED  ] rerank 0.997  overlap 0.74 vs 2024年秋全球姊妹校交換計畫甄選簡章_0314.pdf#14
              教育部學海飛颺與惜珠計畫   學海飛颺計畫  教育部為鼓勵成績優秀的本地學生赴海外大專校院(不包括大陸、香港、澳門地區學校)修讀 學分，特提...
  [REVISED  ] rerank 0.884  overlap 0.65 vs 2024年春全球姊妹校交換計畫甄選簡章_1012.pdf#6
              第1組：日韓東協地區2025年春季交換學習    School Name N GPA iBT IELTS JLPT Note  01 昭和女...
  [NEW      ] rerank 0.159  no counterpart
              08 奈良女子大學  Nara Women’s University 1 2.5   N3 Female students only  09...
  [REVISED  ] rerank 0.688  overlap 0.77 vs 2024年春全球姊妹校交換計畫甄選簡章_1012.pdf#16
              School Name N GPA 

### Ask the LLM what actually changed

Only the **REVISED** sections are sent, one pair at a time. Each prompt stays small, and the
model gets a single answerable question instead of two whole bulletins to diff.

Two Ollama details this cell has to get right, both of which cost real time to discover:

1. **Thinking must be off.** `qwen3` reasons before answering, which on this GPU meant
   *minutes* per call instead of seconds. Ollama's OpenAI-compatible `/v1` endpoint
   silently ignores both `think` and `chat_template_kwargs`, so the only reliable switch is
   the native `/api/chat` endpoint with `"think": false`.
2. **`num_ctx` must be set.** Ollama loads this model with a 4096-token window by default
   even though qwen3 supports 262144, so a long section pair would be quietly truncated.

The distinction that matters to the office is a *substantive* change versus a *cosmetic*
year relabel, so that is what the schema asks for.


In [9]:
import requests
from pydantic import BaseModel
from typing import Literal

OLLAMA_URL = "http://localhost:11434/api/chat"
LLM_MODEL = "qwen3:4b"
MAX_SECTIONS = 3   # keep the run quick; raise to review a whole document

class SectionChange(BaseModel):
    change_type: Literal["SUBSTANTIVE", "COSMETIC"]
    summary: str
    old_value: str
    new_value: str

SYSTEM = (
    "You compare two editions of a Taiwanese university exchange-programme bulletin. "
    "Report only what genuinely differs. change_type is COSMETIC if the only difference is "
    "the year or edition label, or pure rewording; SUBSTANTIVE if a deadline, fee, "
    "eligibility rule, required document, URL or procedure changed. "
    "old_value and new_value must quote the specific differing text, not whole sections. "
    "Answer in English. Output JSON only, matching the schema exactly."
)

def describe_change(old_text, new_text, timeout=180):
    resp = requests.post(OLLAMA_URL, timeout=timeout, json={
        "model": LLM_MODEL,
        "stream": False,
        "think": False,                          # see note above - this is the switch that matters
        "messages": [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"OLD (2024):\n{old_text}\n\nNEW (2025):\n{new_text}"},
        ],
        "format": SectionChange.model_json_schema(),   # Ollama constrains decoding to the schema
        "options": {"temperature": 0, "num_ctx": 8192},
    })
    resp.raise_for_status()
    return SectionChange.model_validate_json(resp.json()["message"]["content"])


In [10]:
import time

# The embedding models are done with the GPU by this point; hand the VRAM to Ollama.
if DEVICE == "cuda":
    torch.cuda.empty_cache()

for name in incoming_files:
    revised = [r for r in reports[name] if r["status"] == "REVISED"]
    print(f"\n{'=' * 78}\n{name}\n{'=' * 78}")
    print(f"{len(revised)} revised section(s); describing the first {min(MAX_SECTIONS, len(revised))}\n")

    for r in revised[:MAX_SECTIONS]:
        t0 = time.time()
        change = describe_change(r["matched"], r["section"])
        flag = "!!" if change.change_type == "SUBSTANTIVE" else "  "
        print(f"{flag} [{change.change_type}] vs {r['match']}  "
              f"(overlap {r['overlap']:.2f}, {time.time() - t0:.0f}s)")
        print(f"     {change.summary}")
        print(f"     old: {change.old_value[:150]}")
        print(f"     new: {change.new_value[:150]}\n")



2025年春全球姊妹校交換計畫甄選簡章_10-01.pdf
9 revised section(s); describing the first 3



!! [SUBSTANTIVE] vs 2024年春全球姊妹校交換計畫甄選簡章_1012.pdf#3  (overlap 0.94, 25s)
     The 2025 bulletin introduces new application eligibility criteria, fees, and procedures compared to the 2024 bulletin. Key differences include: 1) New eligibility requirements (e.g., '本校修業滿一學期的在學學生') 2) New fee structure (e.g., '交換學生甄試報名費新台幣伍佰元整') 3) New guarantee fee policy (e.g., '獲錄取交換學生需繳交學習保證金新台幣壹萬元整') 4) New international scholarship programs (e.g., '具中華民國國籍並在台灣設有戶籍學生得申請教育部學海飛颺與惜珠計畫補助')
     old: 2、團體口試(40%)：由3位跨院教師擔任口試委員，依其成績平均
     new: 2025年春學期東海大學赴全球姊妹校交換甄試簡章



!! [SUBSTANTIVE] vs 2024年春全球姊妹校交換計畫甄選簡章_1012.pdf#2  (overlap 0.85, 13s)
     Deadline for application changed from 2023-10-06 to 2024-09-26
     old: 2023年10月6日(五)上午9點起：進行口試
     new: 2024年9月26日(四)下午4點：完成網路報名、繳交紙本資料至國際處



!! [SUBSTANTIVE] vs 2024年秋全球姊妹校交換計畫甄選簡章_0314.pdf#14  (overlap 0.74, 14s)
     The 2025 bulletin has replaced the 2024 bulletin with a new structure and content. The old bulletin listed specific universities and their details for exchange programs, while the new bulletin focuses on scholarship programs (學海飛颺 and 學海惜珠) and a global list of sister schools for 2025 exchange programs.
     old: 序號 校名/校碼 名額 等級 備註
     new: 教育部學海飛颺與惜珠計畫


2025年秋全球姊妹校交換計畫甄選簡章_0319.pdf
9 revised section(s); describing the first 3



!! [SUBSTANTIVE] vs 2024年春全球姊妹校交換計畫甄選簡章_1012.pdf#3  (overlap 0.90, 22s)
     The 2025 bulletin introduces new eligibility rules, fees, and procedures compared to the 2024 bulletin. Key differences include: 1) New eligibility criteria (e.g., no open to exchange students during the 2025 fall semester, requirement for full-time enrollment), 2) New fee structure (e.g., application fee of NT$500, deposit of NT$10,000 with refund conditions), 3) New deadlines and procedures (e.g., specific registration dates, new documentation requirements).
     old: 2、團體口試(40%)：由3位跨院教師擔任口試委員，依其成績平均
     new: 2025年秋學期東海大學赴全球姊妹校交換甄試簡章



!! [SUBSTANTIVE] vs 2024年春全球姊妹校交換計畫甄選簡章_1012.pdf#2  (overlap 0.83, 15s)
     Deadline for online registration and payment shifted from 2023-10-06 to 2025-03-10
     old: 2023年10月6日(五)上午9點起：進行口試
     new: 2025年3月10日(一)下午16點前：完成網路報名、繳交資料、繳交報名費



!! [SUBSTANTIVE] vs 2024年秋全球姊妹校交換計畫甄選簡章_0314.pdf#14  (overlap 0.80, 23s)
     The old bulletin lists 2024 exchange programs with specific universities and requirements, while the new bulletin (2025) introduces the '學海飛颺計畫' and '學海惜珠計畫' with detailed eligibility, deadlines, and procedures. Key differences include: 1) New programs with specific eligibility rules (e.g., nationality, residency, academic performance), 2) Updated deadlines (e.g., 114 year 3/21 to 4/21 for applications), 3) New URL for scholarship information, 4) Changed exchange program names and dates (e.g., '2026年春季交換學習' instead of '2025年春季交換學習'), 5) Different structure and content of the exchange program listings.
     old: 序號 校名/校碼 名額 等級 備註
     new: 教育部學海飛颺與惜珠計畫

